# Cert Prep — 07: Pandas API on Spark

**Exam weight: ~5%**

Topics covered:
- pyspark.pandas vs pandas
- Creating Pandas-on-Spark DataFrames
- Operations: same as pandas syntax
- toPandas() — driver memory warning
- to_spark() — convert back to Spark DataFrame
- Index handling


In [1]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

Spark 4.0.2


Dataset ready
+---+-----+-----------+--------+----------+----------+
| id| name|       dept|  salary| hire_date|perf_score|
+---+-----+-----------+--------+----------+----------+
|  1|Alice|Engineering| 95000.0|2021-03-15|         4|
|  2|  Bob|  Marketing| 72000.0|2020-07-01|         3|
|  3|Carol|Engineering|105000.0|2019-11-20|         5|
|  4| Dave|  Marketing| 68000.0|2022-01-10|         2|
|  5|  Eve|Engineering| 88000.0|2021-09-05|         4|
|  6|Frank|         HR| 61000.0|2020-04-18|         3|
|  7|Grace|         HR|    NULL|2023-02-28|         1|
|  8| Hank|Engineering|112000.0|2018-06-12|         5|
|  9| Iris|  Marketing|    NULL|2022-11-03|         2|
| 10| Jack|         HR| 59000.0|2023-08-01|         1|
+---+-----+-----------+--------+----------+----------+



In [2]:
# ── Pandas API on Spark ───────────────────────────────────────────────────────
print("=== Pandas API on Spark ===")
import pyspark.pandas as ps

# Convert Spark DF to Pandas-on-Spark DF
psdf = ps.from_dataframe(df)
# OR: psdf = df.pandas_api()  (Spark 3.2+)

print(type(psdf))
print(psdf.dtypes)
print()

# Pandas-style operations
print("Head:")
print(psdf.head(3))
print()
print("Describe:")
print(psdf[["salary","perf_score"]].describe())

=== Pandas API on Spark ===


/opt/spark/python/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


AttributeError: module 'pyspark.pandas' has no attribute 'from_dataframe'

In [ ]:
# ── Operations ───────────────────────────────────────────────────────────────
print("=== Pandas-on-Spark Operations ===")
import pyspark.pandas as ps

psdf = ps.from_dataframe(df)

# groupby — same as pandas
print("GroupBy:")
print(psdf.groupby("dept")["salary"].mean())

# fillna
print("\nfillna:")
print(psdf["salary"].fillna(0).sum())

# value_counts
print("\nvalue_counts:")
print(psdf["dept"].value_counts())

# apply — runs as Pandas UDF internally
print("\napply (row-level):")
psdf["salary_k"] = psdf["salary"].apply(lambda x: round(x/1000, 1) if x else 0)
print(psdf[["name","salary","salary_k"]].head())

In [ ]:
# ── toPandas() WARNING ───────────────────────────────────────────────────────
print("=== toPandas() Memory Warning ===")
print()
print("df.toPandas() — converts Spark DataFrame to pandas DataFrame")
print("  - Collects ALL data to the DRIVER memory")
print("  - If data is larger than driver memory → OutOfMemoryError")
print("  - Fine for small result sets, DANGEROUS for large DataFrames")
print()
print("Same applies to:")
print("  psdf.to_pandas()   — Pandas-on-Spark → pandas")
print("  psdf.toPandas()    — same")
print()
print("Safe pattern: filter/aggregate FIRST, then toPandas():")
print("  df.groupBy('dept').agg(avg('salary')).toPandas()  # small result ✅")
print("  df.toPandas()                                      # 100M rows ❌")
print()

# Safe usage
small_result = df.groupBy("dept").agg(F.avg("salary").alias("avg")).toPandas()
print("Small result toPandas:")
print(small_result)

In [ ]:
# ── to_spark() — convert back ─────────────────────────────────────────────────
print("=== to_spark() ===")
import pyspark.pandas as ps

psdf = ps.from_dataframe(df)

# Apply pandas operations
psdf_filtered = psdf[psdf["salary"] > 80000]

# Convert back to Spark DataFrame for further Spark operations
spark_df = psdf_filtered.to_spark()
print(type(spark_df))
spark_df.show()
print()
print("Index column warning:")
print("  Pandas-on-Spark adds a default index column")
print("  When converting to Spark, index becomes a column unless dropped:")
print("  psdf.to_spark(index_col='my_index')")
print("  psdf.to_spark().drop('__index_level_0__')  # drop default index")